# scgraph 4.0 Showcase: New Features & Multimodal Routing

This notebook demonstrates the core improvements introduced leading up to version 4.0 (since 3.6.0):
1. **Binary `.scgraph` format**: Compact coordinates encoded as `int32` (10⁻⁷ deg), near-instantaneous memory-mapped CSR reading, and Hilbert-curve spatial ordering.
2. **Instant header inspection (`GeoGraph.scgraph_info`)**: Inspect file metadata, node/arc counts, and attribution in milliseconds without loading graph arrays into RAM.
3. **Reproducible local pipeline builds (`GeoGraph.build_geograph`)**: Download and build reference networks from OpenStreetMap, World Port Index, and Eurostat sources with full identity provenance (`metadata`).
4. **Mode filtering & Multimodality**: Extract a single mode (`modes=["road"]`) from multimodal graphs and compute door-to-door intermodal routes (road + sea + port connectors).

In [ ]:
# Install scgraph 4.0 and optional dependencies (osmium for OSM extracts, folium for mapping)
# !pip install "scgraph @ git+https://github.com/connor-makowski/scgraph@2ec389ee7a385f40912bae18910e352bbaf59ed1" osmium folium
import os
import folium
from scgraph import GeoGraph
from scgraph.pipelines.build import build_file


## 1. Built-in Networks & Zero-RAM Inspection (`scgraph_info`)

In scgraph 4.0, published networks are automatically converted to `.scgraph` on first access.
`GeoGraph.scgraph_info` reads the header and metadata directly from the file without allocating memory for millions of edges.

In [ ]:
# Load the global maritime network (auto-converted to .scgraph on first read)
marnet = GeoGraph.load_geograph("marnet")

# Inspect header information instantly without loading the full graph into RAM
cache_dir = os.path.expanduser("~/.cache/scgraph")
info = GeoGraph.get_scgraph_info(f"{cache_dir}/marnet.scgraph")
print(f"Format: v{info['format_version']} | Nodes: {info['nodes']:,} | Arcs: {info['arcs']:,} | Dtype: {info['coordinate_dtype']}")
print(f"Attribution: {info.get('attribution', 'N/A')}")


## 2. Global Example: Transoceanic Maritime Route (Rotterdam ➔ Singapore)

Calculate the shortest global maritime path between Europe's largest port (Rotterdam) and Southeast Asia's hub (Singapore) via Gibraltar, Suez Canal, and the Indian Ocean.

In [ ]:
rotterdam = {"latitude": 51.95, "longitude": 4.14}
singapore = {"latitude": 1.28, "longitude": 103.85}

sea_route = marnet.get_shortest_path(rotterdam, singapore)
print(f"Maritime distance Rotterdam -> Singapore: {sea_route['length']:,.1f} km ({len(sea_route['coordinate_path'])} waypoints)")


## 3. Rebuilding Networks from Raw Sources (`build_geograph`) & Identity Card

Networks can now be built locally on demand from authoritative raw sources (Geofabrik OpenStreetMap, World Port Index, Natural Earth).
Every built graph carries an immutable **identity card** (`metadata`): pipeline versions, source timestamps, hashes, and simplification steps.

In [ ]:
# List available reproducible pipelines
print("Available pipelines:", [p["name"] for p in GeoGraph.list_buildable_geographs()[:3]], "...")

# Build or load Denmark multimodal network (OSM roads + ports + sea lanes, ~1 min)
# Note: If upstream NGA is down (HTTP 503), scgraph automatically falls back to bundled seed ports
dk = GeoGraph.build_geograph("osm_main_roads_and_marnet", region="europe/denmark")
path = build_file("osm_main_roads_and_marnet", region="europe/denmark")

# Inspect the identity card (pipeline, source versions, and assembly steps)
print("Pipeline:", dk.metadata.get("pipeline"))
print("Executed steps:", [s["step"] for s in dk.metadata.get("steps", [])])
port_src = next((s for s in dk.metadata.get("sources", []) if s["name"] == "ports"), {})
print(f"Ports source fallback: {port_src.get('is_fallback', False)} (version: {port_src.get('version')})")


## 4. Mode Filtering & European Comparison: Multimodal vs. Pure Road (Denmark)

A single `.scgraph` file contains multiple modes (`road`, `port`, `sea`).
Using `modes=["road"]`, we can selectively read only the road network without touching the rest.

Let us compare routing between **Aarhus** (Jutland peninsula) and **Copenhagen** (Zealand island):
- **Multimodal**: Uses the direct maritime ferry link across the Kattegat bay.
- **Road-only**: Must take the southern detour across Funen island and the Storebælt toll bridge.

In [ ]:
# Load only the road mode from the same .scgraph file
roads = GeoGraph.load_from_scgraph(path, modes=["road"])

aarhus = {"latitude": 56.1629, "longitude": 10.2039}
copenhagen = {"latitude": 55.6761, "longitude": 12.5683}

route_multi = dk.get_shortest_path(aarhus, copenhagen)
route_road = roads.get_shortest_path(aarhus, copenhagen)

print(f"Multimodal route (road + sea/ferry): {route_multi['length']:.1f} km ({len(route_multi['coordinate_path'])} waypoints)")
print(f"Road-only route (via Storebælt bridge): {route_road['length']:.1f} km ({len(route_road['coordinate_path'])} waypoints)")


## 5. Interactive Map Visualization (Folium)

Visualize the contrasting routes using Folium: multimodal path (orange dashed line) versus pure road path (solid blue line).

In [ ]:
# Initialize Folium map centered on Denmark
m = folium.Map(location=[55.9, 11.5], zoom_start=8, tiles="CartoDB positron")

# Pure road path (solid blue)
folium.PolyLine(
    route_road["coordinate_path"],
    color="#1f77b4",
    weight=5,
    opacity=0.8,
    popup=f"Road-only: {route_road['length']:.1f} km",
    tooltip="Road-only (via Storebælt bridge)",
).add_to(m)

# Multimodal path (orange dashed line)
folium.PolyLine(
    route_multi["coordinate_path"],
    color="#ff7f0e",
    weight=4,
    opacity=0.9,
    dash_array="6, 8",
    popup=f"Multimodal: {route_multi['length']:.1f} km",
    tooltip="Multimodal (direct maritime ferry)",
).add_to(m)

# City markers
folium.Marker([aarhus["latitude"], aarhus["longitude"]], popup="Aarhus", icon=folium.Icon(color="green")).add_to(m)
folium.Marker([copenhagen["latitude"], copenhagen["longitude"]], popup="Copenhagen", icon=folium.Icon(color="red")).add_to(m)

m
